# Baseline 3: Pre-trained Contextual Embeddings (BERT)

In this notebook, we evaluate a pre-trained Transformer model (`unitary/toxic-bert`) to establish our deep learning baseline. Unlike Naive Bayes, BERT utilizes attention mechanisms to understand the context of words in a sentence. 

Since this model is already fine-tuned for toxicity detection but not specifically for the *gaming domain*, it will help us observe whether general-purpose Transformers still produce False Positives when encountering gaming jargon (e.g., "kill", "shoot").

In [1]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import pipeline
from tqdm.auto import tqdm
import warnings

# Suppress minor warnings for a cleaner output
warnings.filterwarnings('ignore')

## 1. Data Loading and Sampling
We load the same Jigsaw dataset and perform the exact same 80-20 split (using `random_state=42`) to ensure our test data matches the Naive Bayes baseline. 

**Note on Computational Constraints:** Deep learning inference is highly resource-intensive on a CPU. To prevent the environment from crashing and to obtain results in a reasonable time, we will sample a subset of **500 instances** from the test split for this baseline evaluation.

In [2]:
print("Loading data...")
# Load the dataset from the data directory
df = pd.read_csv('../data/train.csv') 
df = df[['comment_text', 'toxic']].dropna()

# Perform the exact same train-test split as the previous baseline
print("Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    df['comment_text'], 
    df['toxic'], 
    test_size=0.2, 
    random_state=42
)

# Sample 500 rows randomly from the test set for local CPU inference
print("Sampling 500 rows for evaluation...")
X_test_sampled = X_test.sample(n=500, random_state=42)

# Extract the corresponding true labels for the sampled test set
y_test_sampled = y_test.loc[X_test_sampled.index]

print(f"Sampled Test Set Size: {len(X_test_sampled)}")

Loading data...
Splitting data...
Sampling 500 rows for evaluation...
Sampled Test Set Size: 500


## 2. Model Initialization (Hugging Face Pipeline)
We utilize the Hugging Face `pipeline` API to easily load the `unitary/toxic-bert` model. This model has been fine-tuned on toxic comment classification tasks. We set `truncation=True` and `max_length=512` to handle any unusually long comments safely.

In [3]:
print("Downloading and loading the Pre-trained BERT model...")
# Initialize the text-classification pipeline
# This will download the model weights (~400MB) on the first run
classifier = pipeline(
    "text-classification", 
    model="unitary/toxic-bert", 
    truncation=True, 
    max_length=512
)
print("Model loaded successfully.")

config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model loaded successfully.


## 3. Inference
We iterate through our sampled test set and generate predictions. The `tqdm` library provides a progress bar to monitor the inference speed. The model outputs a label and a confidence score. If the label is 'toxic' and the confidence is over 50%, we classify it as 1; otherwise, 0.

In [4]:
print("Starting inference on the sampled test set...")
y_pred = []

# Iterate over each text in the sampled test set with a progress bar
for text in tqdm(X_test_sampled, desc="Classifying comments"):
    # The pipeline returns a list containing a dictionary (e.g., [{'label': 'toxic', 'score': 0.98}])
    result = classifier(text)[0]
    
    # Map the string output to our binary format (1 for toxic, 0 for non-toxic)
    if result['label'] == 'toxic' and result['score'] > 0.5:
        y_pred.append(1)
    else:
        y_pred.append(0)

Starting inference on the sampled test set...


Classifying comments:   0%|          | 0/500 [00:00<?, ?it/s]

## 4. Evaluation and Results
Finally, we compare the model's predictions against the actual ground-truth labels using `classification_report`. We expect to see an improvement over the Naive Bayes model's Recall score, demonstrating the power of contextual embeddings.

In [5]:
print("\n--- PRE-TRAINED BERT BASELINE RESULTS (500 Samples) ---")
# Print the classification report comparing true labels with BERT predictions
print(classification_report(y_test_sampled, y_pred, target_names=['Non-Toxic', 'Toxic']))


--- PRE-TRAINED BERT BASELINE RESULTS (500 Samples) ---
              precision    recall  f1-score   support

   Non-Toxic       0.99      1.00      1.00       455
       Toxic       0.98      0.93      0.95        45

    accuracy                           0.99       500
   macro avg       0.99      0.97      0.98       500
weighted avg       0.99      0.99      0.99       500

